# 🔍 Semantic Similarity System (KeaBuilder ML Task)
### Dhruv Sharma

## Objective
Build a semantic similarity system to match user inputs (leads/prompts) with the most relevant entries.

## Approach
- Sentence embeddings using `all-MiniLM-L6-v2`
- Cosine similarity for matching
- Return best + top-k matches

## Use Case
- Lead matching
- Template recommendation
- Query clustering


In [1]:
!pip install sentence-transformers scikit-learn

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

C:\Users\Dhruv Sharma\anaconda3\envs\kea_clean\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2133.68it/s]


In [4]:
inputs = [
    "I need help generating leads",
    "Looking for an AI chatbot for my website",
    "Want to improve conversion rates",
    "Need help building a sales funnel",
    "Interested in marketing automation tools"
]

In [5]:
# Precompute embeddings (performance optimization)
input_embeddings_cache = model.encode(inputs)

## ⚙️ Core Similarity Function

In [6]:
def similarity_search(query, inputs, cached_embeddings=None):
    """API-ready similarity search"""

    if not query or not inputs:
        return {"error": "Invalid input"}

    if cached_embeddings is None:
        input_embeddings = model.encode(inputs)
    else:
        input_embeddings = cached_embeddings

    query_embedding = model.encode([query])
    scores = cosine_similarity(query_embedding, input_embeddings)[0]

    best_index = np.argmax(scores)

    return {
        "query": query,
        "best_match": inputs[best_index],
        "score": float(scores[best_index])
    }

## 🔍 Final Output

In [7]:
query = "I want more leads for my business"

result = similarity_search(query, inputs, input_embeddings_cache)

print("Query:", result["query"])
print("Best Match:", result["best_match"])
print("Score:", round(result["score"], 3))

Query: I want more leads for my business
Best Match: I need help generating leads
Score: 0.652


## ⭐ Top-K Matching

In [8]:
def get_top_k_matches(query, inputs, cached_embeddings, k=3):
    query_embedding = model.encode([query])
    scores = cosine_similarity(query_embedding, cached_embeddings)[0]

    ranked = sorted(zip(inputs, scores), key=lambda x: x[1], reverse=True)
    return ranked[:k]

top_matches = get_top_k_matches(query, inputs, input_embeddings_cache)

for i, (text, score) in enumerate(top_matches, 1):
    print(f"{i}. {text} (Score: {round(score, 3)})")

1. I need help generating leads (Score: 0.6520000100135803)
2. Need help building a sales funnel (Score: 0.40400001406669617)
3. Want to improve conversion rates (Score: 0.3009999990463257)


## 🚀 Batch Processing

In [9]:
def batch_similarity(queries, inputs, cached_embeddings):
    results = []
    query_embeddings = model.encode(queries)

    for i, q_emb in enumerate(query_embeddings):
        scores = cosine_similarity([q_emb], cached_embeddings)[0]
        best_index = np.argmax(scores)

        results.append({
            "query": queries[i],
            "best_match": inputs[best_index],
            "score": float(scores[best_index])
        })

    return results

queries = ["I need more leads", "Help me build chatbot"]

for r in batch_similarity(queries, inputs, input_embeddings_cache):
    print(r)

{'query': 'I need more leads', 'best_match': 'I need help generating leads', 'score': 0.7616357803344727}
{'query': 'Help me build chatbot', 'best_match': 'Looking for an AI chatbot for my website', 'score': 0.7825677394866943}


## 🧠 Design Decisions

- Used semantic embeddings instead of keyword matching
- Cosine similarity for efficient comparison
- Cached embeddings for performance optimization
- API-ready function for backend integration

## ⚡ Performance Consideration
- Precomputed embeddings reduce repeated computation
- Scalable with vector databases (FAISS / Pinecone)

## 🏗️ Production Integration
This system can be deployed as a microservice and integrated with a Node.js backend via REST API.
